# YuE2 on Kaggle — recoverable full-song generation

All computation happens on Kaggle. Select **GPU T4 ×2** (a single GPU can
also be tested) and enable **Internet** before running. No local GPU,
public tunnel, Gradio account, or API token is required for these public assets.

This version uses native ComfyUI with the official INT8 YuE2 checkpoint.
**First run a 30-second diagnostic**, then try 180, 420, or 480 seconds.
Duration is a **maximum**, not a promise: the model can end early, and a
token-limit cutoff is explicitly reported. Seven/eight-minute synthesis
and T4 memory/performance still require validation in your Kaggle session.

Native YuE2 uses FP32 text generation on T4. The selected GPU runs the
stages sequentially; two cards are not automatically pooled. GPU memory
observations cover both cards. FP32 VAE and tiled decoding are retained.
Cover transcription is deferred until this generation baseline is validated.

Outputs and recoverable stages are saved under `/kaggle/working/yue2/runs`.
**Download the run ZIP or save a Kaggle output version before ending your
session.** Files in a live working directory do not survive every session reset.
Downloaded run ZIPs can be attached to a later notebook as a Kaggle input.

Sources: [YuE2](https://github.com/multimodal-art-projection/YuE),
[ComfyUI](https://github.com/Comfy-Org/ComfyUI),
[official repackaged weights](https://huggingface.co/Comfy-Org/YuE2).
ComfyUI is pinned to `bc5f880dbdd43e426077f42ccec53bb577ffee9f`;
model assets are pinned to `2f76ca75e6ee094169de899cc7fc99d6887e2196`
and checked against the published SHA256. Model license terms remain applicable.

In [ ]:
# Generated support modules are embedded so this notebook works by itself.
import base64, json, os, sys, zlib
from pathlib import Path
if "app" in globals():
    app.stop()  # Stop only the server previously started by this notebook.
support = Path("/kaggle/working/yue2_support")
support.mkdir(parents=True, exist_ok=True)
payload = "eNrFPYly4zayv8J1KjXShKIl+daMUs9xPBnXzuEae7KVtV0sSoJkrilSISl7HJf+/fUBgAAPWZ6XrbfZxCKJowE0+kJ342nrLpjNIuE/LkV/2/fDOMx931s8bg2crWv6559UwEnS8a3I8jTIwyR2pknq5LfCWYRxLCZODG/vhXOSzKePX8+cP5anfSecLyIxF3FONTxu7Drech2rzzHW8eNkIjKr249BHMyg6fEyy5O5QwUGziJN7kUcxGPhOqkYw0MajAC6LIfCmesEMdS4FeM7qCm+LZI0z7zr+Do+i6FEFMHbIJNNcp/bBizyjTELnvMRXkVOEM2SNMxv55l7HS+g4zDDWchEJMY4Oug4z2Go9DMLYORhPJPALOM7fEjFPAjj0kwhbHJepimM0veny3yZCt/H6QPooY044RnMcBzy7XQZj/MkiTL9JoyzBcCin/+TJUXxJJPtL4L8NgpHqvFzeNSF8nAujC7i5XzxiLMVL/S7LFnGk2kI8w3vs2lRFZFDdpEFUwFTkSVp5tF71VmUBBMfK+MM3Qv6afRHeODNcbr9Oa094o5slD/m4lvui3gMZaBxXDHVNiLc5SmtlaxhLKsnEeVRlW5dxw78LxgngAjh2E+DmJEHsAIecepchUX++DaMJq4zCaFI7vIoFrdBRggIv6k0N0jDkt+y26C/t8/jvY7bONCLy+PLU2foPK3w6TqeiKkzWyx9wMw8a7UH3EgqAAFi54qf8H9P11sTcR+OxTXsjtB1rrcAk5NxkAN0s3CEb2mivfFyEnhzMU/SR18XaYVtZ9vpv36903WLNqGRVGQivV/XhirR2MRCBHf+WmCCb/4agFZFa0hQQsBihxajZbTBY/dhseK81W5zlZtiCh9gYwqaxGUxi8ZStmjer2ASQ0AEaPjxeusGILjeYqLh0VYBsqTK8Wso1C46GYlZGPvpMm7Bvz5uNdUT/oZFxUY8RI6sKCEXNElyKIB7rZVknojvwzSJoZ8/vp72/S9fP/lfPn++xO4AUbMkuhctWVMDDNUtdGxhmy51De1gf+GE4cVq4RSoJWB5UX2blvvPJWCwHG3bCzNCTj1jBGoQZsL5PYiW4jRNk7R1vfVlGTtz2CbONEzhvyPhjAHtYRmd0SNxAMkexlGI23VLwgC0F/qGUsNil6wHSMNOE/qPoW7iefiwRjgNx8yaJmEWzFIhMucBKDbBiBtz4sgOdV+03t44EkHasl4tFxMYYQHuUP/iOR/if1zJc4ZPhDGAfIj7tBox0HtEqAKVBrDpV7KPjTHdGLhRCDdl7tPOkzuL6Ucom7e3Q4HBiCV6HHrVJa7QuCsTDXOLTEFN9ek3MV7mgmYUWuuMkxgpMrFmQAwYl3BmIhYsIuhplgStbhcW0CUjIjV+Ub8FDBf2XBApUJlRngQLZJCKjwIjXyyQvfKA3zixAEoPfYYAWwqcGjgo8Ps8mU6JZy3HY5FlWhihDRzOoDFsc6i4qKffFVAgpFj8fzTv9R7SYJGZJWjTFqOZtEBAmLrO6yCdAXt5/fruAX+ZC9sw/8a8qeYb2irqjJA5wxg07N4ojCebVfOCxSJ69AH2YBnlWcv4fo+7LYN2J+E4b8nS6WyJ3DmrlPMWyQJQBTvVCID/I64IjXApQAN6QSigivzgnCKTzm9xOW8FrEcICzYOoiB15iIPYE8GDoqdyZ2InbNfYTi82CQzOErmKEOEKJc/LrAvnJw8bVXeG3CK6RQlunuE9cleD2C7ati4obkVt1wm5T2DJTQ/AZSCXq6MjzdGPYMJ3olHmmgUNVoaFAM8EE/zMGY6NyxvaaCqMOlPNLMrn6ZJMTdjZ4sMxgCVS9B5M5G3SCiAzz6KUNb6wfZKmZzjALyFSKdMp0RqIkuePpawGBBcdonScIu4IL9AeI3xeHEA72rZUkEFcUgu9AJbMGBwCnnsubZdnF6XMbFdbRwqwjRSm5fpUtgFRJSJDQHaeL/WdPwugH7sEoVM2TIGJMdS6b/UvCkFrUUWjWtaGKrDPs0hTSHJ+k3N3VT3Du8NBhe3RiTiFj+1kU1q+El8VA9uXRsinhB3Haj2/Cich8DUEdOKdcAF49K8EWBIda1JzYA7VgtBjBs+xBOCtQbhnY7aELWtSroyqFIgbBpEfvxmSP6lRlb2Y5mhl3aXpKxDpE8j2Mm0z/RE1CBtHW+vFOKBHP9yguIbyJ4TJ8wzSXppwhXDD2KHl8RzLmlBnQck3SRxvXFMymM3HcYoR2YsTWA/o+UESBAS+AwtB+GUxUvQ/BZJFhLFk12m0KFIscu65qszhAyWkW2z2YB9QQaMRZpMlmO0biSO3CiOQuF2La+u7EeDsXwbi0Xu/AKtnNJPHBBIJPB+8MwOU/Ioo8DwemsaAIWc0C5FgIfI1KAd2Er8xaeqwzpKtw6ZaCosmU3JMVU5DXcHLQosQ0VS+w4RCSS1afitoGi0YUqCUq2QRKo7EJySMt+yKAw+cA/tDciZbioKHpNlzoJDLUHjNn2brsmOcKtPU+A7FYJXS4gIaqJCbIyoK0TCV4ASf0GdrgJUpvdIqwhcx9eKRXbVG9zc1LaDgnsYL8McVfw/l8AUMqCiGUhQkrQitNxI2/nZ6X0/fVrANs1xQ11JNRE31o1zLGeYrWMZaEfc2+oNL/zwif6sEMWn0TK7HSJLbptSTJ1sXI9EazD6B+efQiwcRFIg8JkTTgRQGzTkOQ+3QOlAFs2cuTREgpi5jEgoZTMS2hJnQRiDxCmJDNCvIIe9aFijPKXRCMQHX1qlNBQwPuL5CtNra6JQUVWPaou21zTjlXvXks468PVmMHf8y8dS24wxMIuePF+zvWkXzYPGf8ZRkGXUCjCAE1ZnFQ2jb6B33CaTgmqdfTr/eulf/nF+etEaR5YmJxHsiQ0boZQp4HEchQv8CdL1yYezc5gnog3KSiQ/XVx+Ofv0G35crZSq+eX08uuXT9wdwG00wN/fff10cnn2+ZODzF+wdq5Y4gnQtd8+f/mDvgXLSZhsk7mdpwNLFaOSVeUuQnhdp2zmIlWxagSrjL9F1dvl+T2H7QUUBhj93z6/JPXYc4ivgSCOxVm8QAI+oDVf0bzXLMe6GS/W5e+Ycyz1+eslDvLT519PDWSsXwsYGy9Ijc0AvnnA+cNFRVeqMZN9SkjIysYoQD2A3KGEG0uUqVESgDViHY9mue0xscd90oI3NcuPEFdW/wKkwQ8B0tXsb19+Om1h1gnj/ABL8OnyuUUtSv1/LKqE2LLvkfEC7RP87coY1403AcI8vm21vSlouzn8HS+W+F9k47NlYrNdVIOoNQ+o4BwNqTtkNVlCay35JbsNFuJq0L9p4/dWz3X2dzdAolNl1E1i5uRORIvqXPXc/V338sZCJi15o+kyzKZ4qKYgaHtBFDXg7TscJJDu8wRkB9W1lhh4WlJTOI873LaEJnsjT/7YOFhCcTwBqzWdXG/J6p5xgmUqGDnKBmlA1nhsxUPzsp8tpyBzAIS0NUh+MnrTdqshneIkDzEatIRPPNyXLUaszfaO+l22Giv1UeLb9ZYhcemjs5aN+wzwyiULlwYVKJ4CYah+GOAlmZeKRRSMRVHFpcEZhdYbEfS04aFdOC0M+m7VfsZnYgitcTrWou5c/szjRlnUQtX2d5vZoKgaN5ZVv03zmzHU51UvNVjSb3GQ/MJnSX9o7a9OD/oHNX9oqPpGZ01yc/1BqJffpsmDL5Tu6IdTH/YAWrMB6Xwyd6fLBZ6v1VBliShVyvwhCSb/Lcr8XRLO30uam0jwi6WbLFmCOKGO8WpQzrSbVk/yiBbI/WEdicl21+wjm6KqjxaWwwOji9pgN0jW1+2Lao3nqf8FHaJJKNnKMw8mgk/ZAmcSTqfwDrgBHVUukHrXUd7qiJsILp4FlukEDsyYg+8Gnw0laGEKUwYY1s8CVwKkTMwEAtJWSa74rA4owhjNee0aVi4bsFi5NbgqmauOjs8toL3vYqXWBJzF90EUTuRhqOTc1AW2z6ZKe8FSGCOwXXvRrENzazyqfKMNP4ns82BVob2hAezpboBtXN3dkLnjDm0draIceUaUbTdthIxKQs21tB7NtAJ9S3xtFVb7WR4c8SxsQMIV2a1j0bUMHl+ukw5QNljVy9XH2MLfr1QxXES4j7/+evb5ObqtC/2/qEnYiDmuB5gYQJE5GgbxG6ytetUsUFvIrIrXSdH6W1WO7r9UjM6Ab4ukABhE6X5ZkpZGIDSlqdEwWiGmlA4w8bwNiwM8u4fdbhehLsELconzlkq9ANYgdyIRAPcioMn+6CRT6MO5e/+XI9HFPKlkqVd1fdW98S4tjcUjB7NWVWGIFwWJI3n1pbrC78enteoBAbleORDBHQDNWAFwBKNMgjAPvlnCW/bnMkjJ8Ax8o6Ur8OvWKErGd9L+OITX9B1ULA8Lt9v1ZyLsGUJVkVxh92kaPIIkEoU5gQHCPIDRY3s2weVsb9NCts1G0znC1VIgbtNaeFn4FzCY16+drrdnowsUf+v0RGf/eXT4lQjkhKcSTaH6mDB6dDKg+XH+RvNY6fEY3AOvRSdNa6p/cM6B+iMLOj/5CDJvfI8WIvQsJX+/KCS3jjfk5JkSo3bQRyGEztHoiqsYheMwjx6NE3/6Aiw0jFs9r+vCUI+OjmACcF2JEdAC09kgfLdFE5yzqxvbIY2PjafznDhNS1IrWM5gzDT73YfjExiW6+hvgPD86V/Hv8OXMtr+ncL9WnUWQbeLfo/+SrQU5bts6l3g6cO7MLJ0RaCrOFyplcPCD9nWPb4N4hgmeth3m7DdmOh5kA9hlqGd5Yj2zPUWYIXf30X+jW47OQgN85rTO1wkOokt/Km6xvZweXMM6kHgVtmkReWvqKkBN/gT1b1xXhNWleZkI8WZ9lc8TZBITD381aorMaVCHmuQSLeL3Y2eYvitmF78Tn/VJzXR+KG/4WkvUnUkgbcgiYnUmSTQL1JeWIbxrTMx93gFG2iroLuQQN8iJQ+blBF2fYyWmuF6XwHpcKKK2zK04U+hCsjyhqsA7WjdXchj+IT8iXY4/jL40TJVbjMG9aS53ED/L/lksDVA7Xk2Bvi0ndjVrM4boekg3GyC/LaCyIe5LTcBw6PTFeXZYFbjA3h/Gd/FIEMiPSq5gWBZ9FNUk2AeIKp3ZIExpYqBo46tFI7hu4b9DHV567JYq3avq5yE+ejeZ96LZfAtHYPM+dRznjU1zHV83IXkNwF/qVlERGoJfzRVBoYZzpdzX/p+4tl4MfSqhgxayUyqx8E3X02N7a9ltR+LIDUal74oxaw6Pw+RC+0BGXlBb6tSdyUjkvH1BcoIYYHSIVCs9k8+HF9c+B+Pz8/PPv2GcvwTqAee7yPv8P0BcOGMKCz+JVVLH7oVx2Vu6XjHLRv83bKdybVVF/SOX1XDQpTPvhUTcjEHURB1b6Q+Ih4/dqapAMEgzcNpMM7J9UU7+y9SXAwQTQp3zxfFWNwGGcZK/F8iKoA1qCiH4y+XZ++OTy7930+/XLBi1CvcKqTLHynhil1xoMNE+ZZPQFzOuATwSdwRd+Ixo4NxeBaLALAnSYkqubzxBiSZoN/9AyxpPOTTWes4XI7RYztKS3bp8V+QVL1b8U3CZrgSmxZhHL/rWIBLmYQsZQbDI5kDwER5Z34H0kqLH9QYxDfAID+5M8/6pxPXklvUjEIDGf6WzhbDQnzp4Nih8aHRnWzMdk4ksQb4+HSSADa1phMpzDQLHHoZ9CpQKRf2BiBjDqJOZV3qJ78kgJCHQ6sqYUyzx3jcUoVQuE4s5eFZIWSKvhGROWhak+J0wFvGURjfteYhC5rF5Guv8cJYg+3aMSpGzEOx2B5VoRPCttFQxZKnm8qWEdpES6iovNlplXCBGNGAYYya1shSn0J01IuC+WgSDNREI2StQyDFvW5/V/6BHTJC4lDRLwkuxf+pWXvryAK1G6QuWIPcWwfVkBD8XbEXyy1EH6UwXykj1WXC8zADJhSRS7yfJ9wmiojczpA6XB9HcaxoKFWhiI+RQH/4cCLI9w8kEcfw3bcnY0FkT6914bPKq6Zd0utcVjXVeERUsl2trrekWog8tUxCSWJRLdOpku7GbMHwziKDa4Pb6ShJolYBlWxiVQ0n4iE9vX4tIXatky2JDPJbe2VgheGqXJkW9tXTe4J03mF59/FXZa2m71hQ+dxLGFySfAsk0eX/oV3LuR5hiGyChWs92WTPKs/3OgQCkfckIR0VG13G2XIhfaSU56Tmy4pbmzE/FhTGmhIc+vkZANjCr/oL0fUjA+XGUG3In0tKa9v6NESxB3ZfHdrAlBw+5aYLgVyCpBUj6ZUIjcjVLiz11Bb65MWPLZSLW/BRqiigznP8LNIpiY7PDE3Z7iuTWQIPuI3r3IYzIh6gCff2evu7O+1aP2FSJFpQ4nBvB0oe7u73e8UoEW4EFlp03gLjdd5yu2XAySqK47NnzdDS9Lhxh33vOO+TcTBaRigFQIes8Dgq7MqmRWpBGKArCxgj4qjsMMr6t3Ii9FUrMshp2N/dO9jXMX/hX2iQbakIqI5dC553yDTX19Mptfu3To89nKE+/F5Pks9BvlyQ2fWeFHQNsgKqPPKrFpkuXDKAKSsG9uXK/mEt6i0mavBkJLypkcZ5r1jC+KckF7Ckdx3YzLgaVoC455yRDKw3YMgB2EDmx3eoXDsaDvTLC8cvltKpFKrxdOYhMl0sw0Alt/hUhEjTH5QxtEtArWw/N2X4jQR9EMJyOlvQgdK3yzyMjLhpEAeK2GxQlaV1r3j1mJUCseXDMo0QYnL0Lr9UAY369TKc6KmpRD1b0c0qnLkIYTYlNGzk5PPHd3/48N+PZ5d0UjMa700PD7uT0WSyuyN2+/vdg4Ppbn88FuO9ndFoD56mQhxNcQk/gnr5wf9y+vvZhToE6k8P9sfBwZ7YF6J7tNvbP5qIw6Oj8fhgOj46muwfHh6Ifu9oH6ufvD89+ef557NP3DPF5u+M0A56iN6d92mSlw+NuceL98cwBqp0tD8VvaOjnYODne5Rt9sTk8PxpL8fHO3ujgIhxOF4vNML+t1RcDDe2Z/0JjvjbnAgetPd3u7RZLRvNHr2bzyC2jna7x7twAx0r+PPv1ycfgG2aGtz52fnxgsY8p63g83g+3+9Pz39gGcL11uLcNF5MsquOovHnU4MTLsDVNd7uI1Upa9fPvDp2m2eL7LB9jYuDqZJyG+T+DZBk4OXpLNttam2d3e3d8bbpbCI663JQe8AZN3Dw71doGNHvcnezsF+H8a+Jw539sf9yV7/aGcs+t1eb6+/M4aV6e9MpwejaXf/EFoDKqKH0GbIrIne2zva3dnp7cDcHvT7o4ODg939/d643+vuHnUPutP90WjUDZAXQYuAB+PDyd7kCHrb6+2OJ+ynIQXnZD4HJb4lg6aA/D5MhijOgHoY38tfYw4BHVrexrZyJwlisdE89Ou4QnvlN6aB35D+YTc33An8y33AvzL4n3W458zXxB1Y25OA+ckyB9ljKB91RDQFgBggnVBCinN+IqJfDQhB+Y2bGdR51mPYh5flEyAPxBf5CfrHJ1Jrrjr73W53AKNE3BkCnZHl2yXeU9L58iCMpJgKKqLIhv29Z/R7Ux8hXT5r1Rw1X2+BWAFSxcx5FKCYS0z9wbkQ4s4ZAR4/BOkkQ9PjrcAUI0FMcjPFhCbLNMNDR5jAGShD8JsmGgWqxPly/NEzdEYCgxTHdSqjVAwz6BzZYL9d/wmP3bpK3/dygeeRwOR7OLXtds0YrzGvh/efhPhwoXu2PTaut4icoz+ZVODRtZbO+WiuW7Bq9GOgUhD8j+ZmIKckf4lY6ensC3CRxLOTJJ6GMyWg5I+RIL1XyoWPaTjOjBemwXHAB55I4rrqVAxPqKg4bfHpMorUUoHwWHwwXvowU5MBCbdADGUzyjpvfzwo5d8wPx4U8GGjMm5LftztAn+QAxQL/XqnX+ptPJ0Vg9InfcCRhY8Sjqq31+sbX5BZRsFCfdzfldgeBbGfxNEjq4hFlGbhjOD7C6DFMmsMuSTUuGsbKgOW8GiJCKe01lC839yxG8jHParoAa857g0QGuscccsAME7UQMAfXg4ChupFIidzJGXF6eQgQqKvETVYB9OcjRfytJ96NzFTw3WIOkjlM77cPew+D+FHPgUorPLKvnHoedCA9GfInNZyAeqDI0DRyVGIXuYgM5fhZjAw5wHLtRRsQBuEvU2jZPLIv+kkaLXBBAKKUTwMtapgwzbRfRfbw3mg1p5fVgoZUGtKmhm91mjcpI412YV+OdEQka6DNnbdmP4UUHMiiJtni1RPPhyjUHAFrUI00ghsUDcDT0HUIYikIxNNHzYpJ3Ah59iCj5QhgadrcQEPkSNXbkaTfsl3FtlqV8Pd5ZzDN1Phl4jcZUSGLt9iCpz9ncYjW9sNJxjfcjU94TE2K2YCz4GdrudRa51eef4LDLBJah1wO31rmxWF8TUxu80WRIby1oO60/c8aqsZUiLudQAWZICK4NP+7gYw6eABqlUPF5CC/d0NaZTJaDSN6hXAGZ/xZX8D71R1jH3y7jcNYA9W9Y3Ukkmo6SF1Quuqw6lfyuC2ijnUzK48j/ZX50eYVDmA/V09guI7gt/dPaxiqUVdTAba3KEsgH3WtmcvsVXlbRkutK5sQB++YoQ5VNpWDc2XUR6iRyYKkodvnOI9z/lIoMXrNoim5HOEdWmWdSKoZRhNfOayLTzA5ZRApKxI91EkQ7ZywpaNX7AqYt3x+ZkzS4MFyNYUyIU+T4CIQMYekvRuigCw000eROhfTbtJ5lcxM9coP3X7iFAmJ7JOoiSaUKcci9LT0YkgQPrFyf2Jdt7GI1uRXlAaP+ZnbFJVFe8WOZ0U43OhsK9WK60OGPNhrBQBcXW9ta8CqyswlA6Mazo3QwvUb9VxKVeH6q6/trviQLtuqDJk8IqmzXV6N+UAzjIMUiYmz+l4RsFzFlPEl8QU/2EzxfVxfWooO2uH8puM9wVKXB1LXTR4Mbo+VujecOaqx4gPVRBWKaxebylRTr6XMmRds8itdLmCrXLggf6ADy77aRj8pvhsva7tB08zcbzyDKjrHdDRTrLwF/x8pJ7v8NlOYWckAlmInIKGfdAZgyh/JLdrr9vdYx8Weuc/hPEkeaBP3XJDq1JcPK8/LxfO6rMYUFrtH5x3SD2X0uuRwjAzIkpAq4BTUNotYGsg5rBDBJC0mI7ppHnYK7d3rL7w4QgWpxye7LwhQITkRmGql9hXCMgYZQm2mXH+N68eIXfXImThHFKzt2T0LkmtdbjYOKu7TbNaKGy1OToI5ip52NtoT33EmX9mV/0XdlR5mqy9Zcuo5YqV3dZIf6RjmVZgKi1ZPkrGBjU9yEq7kbxw7d24V2lY703cU2v2Yp8wZDrzKUqiOn74ZMb1rV7Cck6BnT8S3yGec1yEZVjYaibeIJxRzGCE54skmRCsdZ0fNHb+zwv273wWschv2OREjFmcIOdemGDxl1jMgpov5XZlEGM4D2aq6L6BuBYlN7UfRmuQqQ20hqdK+7AyBjZIb1YtPFxvTRZzvy8zJmKyoSVNBn7JZnN/GYccuEFRMiJO0Fopm6uut5ruw7Ub2nBQq1vmImiHp6Ikzai9cmDTKNX1UWPXvx+fsgs9IdilSuPTsOhlQA6LNbkPhIkHfXqppWO9HPoNmSNY1LU/ypfK3aE0kl732Vls2io6gognxJwnmxIXIU2cIwYbHpjGNcOu5pIny/B6SwYLbaPADFo9BRBhJ6854rZrik6kOTzrcmOV3NhBTdeiEwM0vA+N/jBeTmZ6Ns9kqMC9iO8rhfFlpSQf96iyVA/LjsJ4mz9VahDDLreNL2tLysGuHR4a7MsNUkKVFD9VmoU1kAFL8Kvcljz/gAK2izZ9HAWZ8JdppL8aMV+cORcDJipGVp7MIr2uN07s4CLpkCIj+QgNfDrr9H0DFZR/ouU3KNsHPD7/4/L950/nx5fvZVCgbPQnBzP7kiOyWMBTXemQBiD9I8xPbUVNLG2emyinCFaBiGrpKhXev/Pff/3F//Xs4viXD6f+5emH04+nl1/+4LpELeoH9fXTL1/fvTv9cvprbVG5ZRE9TcP3Ilz4D7dCRJUFYT31nUC3m8ABPAmnKMVSYZfUW9wsLLiS90DsQGPs3IHSJzxYui4d7GDlChqO4Ae6HQBhg2d9UFmq9lLPU8MAxPVrjrTKUTVcsBRWA13llSiKap5MfXBln+178EinWPI42CXvAJi64X6XjrQAdRdJjBnHNSTq2OtBHXtB8aYQGPJToM2CeyEZ/adVNAjVakLVSlHZhTMpypLF0XBDh7XhKL+oFSQU4FWWB8zzMCPXLQwEmy7RSZUM9DITcVid2WbHWGq3VLrqIVtZ12Yf2YYZ4X7WzkZ90sENpgGTVM6BZTtP9HklPd4BnWBT5MuFHTbKe5ZKmrt2kaK3uPDX0VOZbJlSfbOb4CII+coHlhhABgJO5ATTXKTKkipj1IodTAbXyjb+QTL6Vxnm6s7TxDlnDjeHVm/Ruhk4oxQ1/m056QZNcH5lfyJQASKzyTDnyx+ku9FSZpacY5oC2rqk3iZTp2CbuDUClRIaOk/vgJtZeTjIF+EKz8wZ4zCKEQWdjhRGmV3Tmw6UApzpgCQuOsofQ32T5K6zQL3QLfi4GbhLAgR6H7569cryS3KV65GLXkIuIsd1LMUZ+cUjfoZwBuns/qpnJ6BA96g0r5aViQqLospTmCayKEE+mPBErJlf1TeK61PHVOu8k0lvAQzBdHrxrcDYlQmbQ/GaADptVl1Qq9I11fcRSD3S/g1ex4FyhmEABeFTyUUDKirr8gl7q30ltSZW0kvNuzqtI9UGZkvgrvCYn1bGWi+OUKvxMjEENxpq50ziAlkPqK5bsHLEBShm+ATdPOt3giDY/ibMzLQ/ik2hJLgeEwVUQUo0yWKuBmMvUUzNbcuoqpEU5mDx6LJH20vQVANgLu5/BXfpM53qxNg6unYo5qA6ZxuT3K+4hUPoh31I+RlvGQBqgFu6I5GnQ3OsVCFyd6wxo3Q6cdLBiJFvqil4npAOLXuaxUkqOqG6scYohifsIVvjCWCYIJptxDuf81O2NNT4DlVs0JZ8HyOxfZ/Yg4XFNqErkLaKs3rhDfqlwLDiuukEgVgYnqXEM0wbbDKOAbMLueedWZSMgsiRtAAJAGj4y9mtZ+cspxiRzSBVu6uA0tpY2n+sklNUKgjmVR4c5ME+VcoqX/LU6d00JMtqEFN1fizu0Eq4IrOlnhacmbyxYM4ki3ziSlevePCvblYuE0yawOJruIBPTqvgxJIpTtpenYzA9UwhQaKf1LqpC79I7EOeeFV5AU/bcHHlXlErrm5lWC5A5Z+Igv0zz4SxXaLz8Pan5fz8sSIuKCcCpTaagb602Y2P3iIh3ywog9+fF7uuty7yZMGigszzymptkWx6CkhAdxJgFIycFvuGC0NXINIivYE9OrnNkPlDP4Avy28okKNRv2BKPkWIv3Va6IPfbW8CMc2y8wHbU7evSOz47fwrQkFu2U5Lvtzxet2fql4sLPk/3IZjhG4Wcq6czefttxB9T5S7BblQGNfBSFOi6WuRCoE3BCGF576BhN75ywwKFlul7VFEp7wfyIJYN/DW6fU3Eqg/o6nuSdUbeL3pyvkt/IVaesOxeUVek16fv9GJNYjShFvMWWtdlyyrT5N2aBdi3bPdQH3lIjD3UL86f1IWO/RQtdoyBcdlSmm3qoKI0WIq7kEqTTN5kvv+9PjX+obdl0kV3LchVehjzJZtFtumE4QwBqqNzo4N81U3GVO0IWi2Kxb57bDHj8orGoreLkegxc63ydrW+ZzOttUNa7Id0539mQltAkQVLWDBLEIyEcjp5cl7v3laTUNNZEyczVlQWjEB3WQfnuJMInlSl+/pKeeMbBlnVDEu68OsW0h63pCWFABxfiDRyc7fE6b5o8lyjYnAhrVUAhIGOV5tjFiMRnLkFjZRn2v9DktjV0OW6clu0X16EmIGUB4ZkW4WC02HYmRLMlwY56V00wmzk6qKbDLqMG5KFyNzkvGNeJzNE3/yjBsvjMRflAhJnX6E6ie8HQX8M4ru5yBxiGqUar0dCcFTKTrwkg+EazUcPlUDYBQXalGE6apiTZFO6zUVz1nF/ZTk7zAvDK1IHSgUgFPOXBw/tr7Z3JHmZDhEHqS983EYG6GBlieQ/0lhAnmTtB68kfc00k1QTCstRlnlVeMkRlGNHYFtMY7PHToA/PiuU5TLvBxdR+qaMHMmW57hND7nJ+UubgnRnxKn05HyEhp98kDyVwIWhjoKYzQ6AsLgtZPTcLx98vXXY4fgqjVilCRmacHQGk6hsODXVJspLAIumT3dgcQDrlNTtRyup4CDbYM4Q7EIMG443PX2Drx9ScaXsxks0zQYiw7Q8eGw6+3se+o8UF05ia97IMvUJ+PAyA2C19qxlmydjESDeUXKa666x9K8AdN1aE+6xd2XGLtcDMXllE7+XQhcSsT2lZheHHt0FwZsMRDRgTy90yqtcaVbmPk6SVaLzAWnMd1qKoUpxNZgPBaRvlqtaIUS0qHaYMa1cNuGVUOaPl799AqtH+rJgydKXvfz0Gn1XeeQ+mbx0pTj2DCtNtbPw753iCBwCsxsiGTwpZfaLYZGkZnIfVkMlmkB4wpFcaGd7EYRtCfEVdKeB6ELZJIPkRd8xdP11n0azOVFlAsvTwCn5TV52/L+yi1gQ8EohAlAtwIZEl4HSlEOQKFA7j8l0DDGSdxq9dxd99Dd3227RTJQaAPwlhO/cWEKQ9jByJK7cvX+i6rfm9X9KLwTrbt2CZ20m+o7j5wkJv4kyX1Ohpf7+o7a1p/unXvvCsIyf/ZnwDxZZtp7xqSm1Rt+o5AM09IotjeoIKCrBic/KZMcvnTV/aZ4dCx/Ve1tt0E6eQjolj5DQW+kb0SBaNPfuA3UoV4i2UTLVyopHnP+zAmlFHhXxmBuCp92LPnW6W6ktuB2f1K1VsjIlnF9Er1ma4MCRxsb1AsrIa8Mxd1sRsscA7eOkgJlDjfu7KZ+Yku6IEVQ25FSlDCAJJbF1Svcz69uUGyBB4ku8Hy9RWRuQQKCHEDb4KLGZQb2jGh7mrbOMwez7jRgwAxI6UZoJQRYvHD9JdImIHy39Ys8CuxjufJxNZ0w4vy08C4f8/psyvjDMCOMxk3Wdn61IEWX4rphkYtRtl2YejI6Vi0cfMutfOfBatmiVFacrXzb5cK1zhukV0mYmjIQ62ZludpmKlf9YWbeiLyhtQBBOpUhuPgguKhA69spPvmqlh3jzaS1VESpMqS4ov+LoXDyImxfb/1kGM5drcQNDXM57juog5kgi9c7N+06o689wevJaDF+10AE17GjvDc5sXBK0jRMQodgJqrx3yDTNEzUNTDNLyAjBQ+Amm2EdiONNs5vZZZno9CGR7im8zxlUtiuHGJbp5NhRjG5lTgYRk1lIaFAC43V2eOcTqPr4o5KKE1HHzx8/W7TBJHGQCjaN4jIDM0bOisbDtDAv5CXhZfP5Gu2kwRUDgVPBCuJtWW+Cel5RiQOUZRz65kWEuVQHvlqQ2CJTRCznAF/YK02+cvxlVGpb+QgKgf+N7Wt69bfefESgt5u6kPxMb+SfUhxUuJm1g5pk4ivhAGson6vNhErVB4QfYTBz5UzDEN0VO0XadLaleIY/4QmV0pvoPwTJh4pPsWVZ2TfochFIWKHEsXQKQPCEOFJfhLPyFfdK6Wx5y4+8Y3RdG/NEhOGvDvf6XNMo9EH/P9yl3tmqwG+xETHMt/wG0rEsoQZCjBca5EkEbkvwf57SKBqVn++InMNGQcs0sFHHrBwznYkgRPadq7KhyVzHrwuHH92uqXoYttv7kVHHxhZRoRIGUNeZeVzEPL7r0vtTQcwmFhIJu6SHbM3m4EAKhWWzipozg7NATRW8nr6YkxOMbSfjOmhK21kUmyau6Gcwo24kFQpKP8t3eBOwT9x3rksMvNj3AVfKb/N6L5ae9HxOuct+VysofxreXDV0OYRRq4OdQmZRKDuCuFyPj6sSWmf8Ic2L9PSPK0qt5KaSWa895eX5w3ZKJp1FmWJfVIrtBrA729jD5d8xT/tFAjtq8Fut9tFib7NJmooY24Qsg/K7aHmrXdYi/x/w9GgTBhC1E4iH2hip5csiW3zSSUnYK07DVpLJ5tvta7dlvBonTNuo99CRDHp1c3IgYKUX8jjPy1OdQG/K8Lv+I7vhIdOev0Drwv/cBhBOSe9cpXFGsCr8S+xpzbIns0Ou1M+khlsb+vWB0/Y1MpUhzDpChrPS6KmPhxijZLvI+UnC1b8yEmEXQKzNpwLD2foKvsOK+PanAkate2/gaS8EwXL2DhjUq4dgH/AFzpSoWvqSDUURR3WCXUF9v26Be0SR1PztUZtbOiFhGTKEaRV7sVOv0NhAPwMLK1DnL4zTpMs62hjT3lY36TlsrEvKcl10JQmF8DbU63wqWDHyATpFp7kN2W8lK7jyiM1WJdeteQbbhxmnlP1ddl77JOmF7CARtMxemehnjFUSV45a87QAOvi8tfPXy9f2BtRNR8kZz/D7P5JXNaNJ0AlUVhDJRVJyDwB+pLE4biFVg5JCE3nZqAr1ZJvdTtVRaGBKirKubGOoA7f+CINTK4hk/KlIMsNyBqDWfR1NiGFDe2NnKE3JMUFb+jXOSvL9PtNbSWj/wAakw9GrSex9m4gRWRdbHRt5G1d6OB6dFkXiWaHidWHEj3Xel2sc12MZm2MVG1cdikOaFWFQDnxDovp7PDlBRif3qr3MZeV1rqR27kjv6jG2f6lbvhKKH8USCNSKZINt1eAoE81yLmqRYRmlQh762DU2jzIiqsPkVsOaJBX+PPGOh+WkKjZaK9qOtS+YB+lOK62m6KQtM/oEiHHNhh7eHi4WI5AenXyJSbuN/10vAY8/07xxxAiW5YU+fXLB1oZ17nk/SmfAPNi1qroRZ1FgohZFgmxaPXKB/NIbFrl1GVWFxZlIlJEDU44CmItUTICW9hxzg8nkaj4xsP0LMU6qkIFylIivZThP/QbrxqUGWnQxlP9LkNjanJT15Hjy6oLGyz7aJk9es6/gpD8L+IxBt+QO7/MAZorPzfga/MwJ5eIADgB+koAfDL1hZoWbqEyIT84l6hDasc59voEtIiA5tHVQOQBaqiZnnOm4gWcgO4PotQXb8w2pa1sHAXh3MFU6LfAnPE6OplnirT1FODCnFPoU5Czlx6fk/N0VjMC2mt2/vlCLZoOYKD9u2rbqghgXU1er79J+bBKAhTzMMak45uzSVX3Ada5pflht36fGhKM2jffFkganmv9LoyiOmrdDEKv+3fF/qE/m7w9mXKuvHYd48pNaSSp5F8phIODPsaW4yrwbUr3QTTceyZLG3VUpNar34VoMNB5vDCINTCqVB2LLMJSa/A07gZdp1LWXE2qAxuNmTFOStqcaFt/qrslpxTmrHz5uItNrwcrsjZLpkNhNpScuz6JvGWrLxwI2dlfWuOV+4lxMeozWrUF2RcFku5dWhOdUqNNc1G9ytIne6O5YM8BXFz5UnI2ZK8k1SBnMCb8K4er4a2X4lH5mBXh3jr6W8d41zFWBTQljId2rEzxTQCtLQxzpGfiH0Or3maJzcoYQ2mYVPBWkoYzDNBjy24miD1lbxzCTbzlLyd7MYaN4RVFJScujH8IJ0qNgs0xxR/Q549//Dj/cXL54/sfP/548W+lQnizOX2Xh8kdkhMwo7KH/9lt0V0oV4OemXKkQKahEXi9LbuuKVcOtS7fCiKzOckMR+FE5RYKOdUCIc/AQhDXvgxb/mzIcWPcUTyQ241zKmg0p0OE4rHhUKAYdxXLOR9Wu5T/AgbVmDnLJNvG7/YmfQeLsMNN6u6pv0bK0rylSx6clG+HMjfxxbI6C5HuqLi8rPoFi+u7EetqlD/W7VfM6FaiKXwTSBPhW+s/YDfiOuZEUrOm8EWqJkVe8jUDD4lOkjkXAacHGoGYCCIhputV8dzobhAGUfgXyq1mTBD6JkrJTPtd3okUlBPPYbcyunONrgUDdoECKTGM0aOsg8dh5iXLkZhT2OuwysFKoQ3xfTgJg042D6uzTFaqEkqBDuWNs3tkMCqQ+rlrCE1wqnarppvfTMDYuAb7KH3sYAIL3MQwI/OFS85uLi0ZO7J5GMbj4hDDvzjX/gyvoHabutHeOTQoaV1NksWwT4fuG5q56sKUQAMBvVrP3qrm2G1aJF99qiQMgune7w68/nSlkq2+cUSQYn4V0nwy2rF8pR4jBccpkYZDF14JvP6zIkaUu/l56Oz2u7UJtDGHy7Lf7e0cKhDkYZ9IQzQEAvOBBtgZciBjdPU1DCN0CQ3SUDCgiKgUUI6qZprYgEn9ikxJMoGwZRIUkxqDn+H9BvOSj0TACSyofPER1Fcfp6hw6UGba8nnQakr+En7FpAIaITZRAEWp51Y3lhV9UMdS1VVYUOtUvmIXWRq6mGg0h9SrixKPWSwulVVyNAnYKwfIznwOYe21J5RYCwV4s6o5Y39HuSVG/Jsma6jJTMSHgiCOGCcaqrOysZMc5350mob6SRMxqnelQVqSTRs5nzUEVmQjYBCCVM5srfWRNzRiPdWqUn1POh7tNw1xuTGyW+ydAG25qidbmxQpg0DGG4KZmUzDRCmbVlm+0nPf73xj+62VwsHXEzWaxqbur1JFrvSdW/qKzSvsWzBWGC6t6npOlu8Dad8f5G6I0faNOoqhlNZ16riY674rTanbebrR7YGzebltUcEhn9F036yIADel6n8CVc3bX2JwYYDUInIG3Z+Y8JArXo2C4k1bZCuWtqbhez4vLD2PFEqcltTckc5OC10KeWaeq1F4OrFdpbeWgKeLr+t6sPNjV0VaMapgwgSvpy2dBfJC5DfgsOVXa2Bp8bT7L86drn0MkScMU/dmaxYUllbMa4P/xsQ44INrnUYoRMu8U3t6t7dNUNi8xRoiHOQaP4SBeRq6t01LGRNu9LwWlh94hrCNRO+vFVkzXrUMydd+XmtSMo1hoVNV24YQBSMKCcFlyyRVNC9Ofym9FESLqC37WaCxS3jbd9a5FqDBlqYvnpqrVmFbRSneyxO3zhP1Mdq7apbAh+Vr53lmj4LiRTE6/1uA/DfBfgFJbWV7J8OAWuP6vC+lubRmRJzs1xdc+ZkGYnbtTZf87iJ9SAyrgu8rvdJylI0pK6p2kgqTQiCeZwwpA+vj2hQoKS5/pcgE6f0M+SURVWXLNwGSuys004bRRV2m+BDnZpJkRDo3pvW2IrxXE/UWdzv8BmhPi293lJ5JSmlPPTb1tdQu4b4QSZu/FoWZTSaXbAzwjaLGZ7Dd0Li2ZKtqTYgVK8Oo+T1SOtzhuEm0VaA9QKwLtd81rN+2YoGrNOWveYlfMmZj91D06HPC+AwNc91NPoH5yweR8uJYMUa80GBwNNBIyEGuAJrCGZxgpo3XUeuohIoUvLfZ+eZ15BRr+yFVLoOyq1Ib8RWO1hDRgiVE+pNBIbcB2v2hLwhskEZb+JP9o1ThVrPFzMCvex3S7cXtzfO66cGa8DuYuTaDPMpGC2ap25SDibCJY/fDLGAfJLlEYQ8gdM+yiCvWylYaRtWzkp0Y7XHJUV1vvsA5xPbKY5RKlk+iVWpKyKadyG3d2UdodwQn1CPa4rrIxasgTdtNn02pgevq9htN5xGL+NWcWTYev2a22vbJ5zGvJv2cpmdo+RWbLkyFCJdZQWVYCcA4My+zEZL9tJAI4v4OgM2jl6+3MRQz1A8NovzhVWRvgwwmRGB8IrlLYr9s5day9sUeMzVSoyIpN9hMRpZ4aaBfRyP8yUGamMpgID+Xr1ShkQ1+Fc3ZLTM3qjDQ7yXh62cRSX5wtdFyrUrfEYez5b1qaEaJ+9Dn4JF2Chaq5Erw+a/jr98Ovv024AvxHWiEAQE3ICYYFt5faDJkm2tnoMOJKwnUI4QtCHQnVTBI98ErtQMrw5wSutAA8ek40FqDJs6hpGsAZV8VzAaB4Eh6DA1DwLHMRjF/FL4RQbbirI5Ul5KQUEgvNIPmNJxMim7OUXBI6dmtpU1lcqcP1dsJCSl4xdtVgDtK16G+aOvDrZ9dlOu9dJRg/tARZwgRaZWazxuScyAFgAG7vMKM4ZRiUdjz9WEtaiL2zOqXFBUkwYE6fg2vK8QABNi8yizRJvroiTlkhcHm6Ucj0WGYvLRKRo3Dko3uLiJ4XbQ2IgRNqVIFdObIBRZXeANn3wxN5wHd8LXUwGSZDFEFDL/kmnzkiSnCMriKzS7tfpfkKUUdw=="
for relative, contents in json.loads(zlib.decompress(base64.b64decode(payload))).items():
    destination = support / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(".writing")
    temporary.write_text(contents)
    os.replace(temporary, destination)
for name in list(sys.modules):
    if name == "kaggle_yue2" or name.startswith("kaggle_yue2."):
        del sys.modules[name]
if str(support) not in sys.path:
    sys.path.insert(0, str(support))
from kaggle_yue2.runtime import KaggleYuE2, SongConfig
print("Notebook support loaded. No model has been executed yet.")

**Install and check Kaggle.** The isolated environment inherits Kaggle's
CUDA/PyTorch and scientific stack, with constraints preventing their upgrade.
A dependency conflict stops installation rather than replacing that stack.
The package list and hardware report are saved for diagnosis.

**Setup fix (revision 2):** environment creation now bypasses `ensurepip`,
which failed in the uploaded Kaggle Python 3.13 run. A checksum-verified
pip wheel installs pip inside the environment. Rerunning repairs the
partially created environment; no folder deletion is needed. If updating
an existing notebook, rerun the first code cell before this setup cell.

The checkpoint is about 4 GB. Setup requires at least 12 GiB of free disk.
To avoid downloading it again, set `LOCAL_CHECKPOINT` to the official file
attached under `/kaggle/input/...`; its size and checksum are still verified.

In [ ]:
ROOT = "/kaggle/working/yue2"
GPU_INDEX = 0
LOCAL_CHECKPOINT = None  # Or "/kaggle/input/your-dataset/yue2_3b_int8_convrot.safetensors"
if "app" in globals():
    app.stop()
app = KaggleYuE2(ROOT, gpu=GPU_INDEX)
runtime = app.install(local_checkpoint=LOCAL_CHECKPOINT)

**Song settings.** Replace these example lyrics with the complete song.
A seven/eight-minute composition needs enough material for verses, returns,
instrumental development, a bridge and an outro. Repeated lines alone do
not establish full-song quality. Planning budgets do not enforce duration.

Start at 30 seconds to verify the stack; a cutoff in this short diagnostic
is expected if the model has not reached an ending. Use `420` or `480`
only after that run works. Use `PLAN_ONLY = True` to review/edit ABC first.
A truncated ABC score fails explicitly and leaves its token artifact on disk.
For an edited/supplied score, set `CUSTOM_ABC` and keep the original run.

In [ ]:
MAX_DURATION_SECONDS = 30  # Diagnostic=30; three minutes=180; seven=420; eight=480
PLANNING_MODE = "full"    # "full", "melody", or "off"
PLAN_ONLY = False
CUSTOM_ABC = ""
MAX_ABC_TOKENS = 4096
ACOUSTIC_STEPS = 32       # Native dpm_2 / sgm_uniform baseline
SEMANTIC_CFG = 1.0       # Raising this adds a second AR guidance branch
ABC_SEED, SEMANTIC_SEED, ACOUSTIC_SEED = 10, 17, 7
STYLE = "English melodic rock, warm expressive lead vocal, electric guitar, piano, live drums, 112 BPM"
LYRICS = """[Verse 1]
Morning breaks across the water
Every road is turning gold
We have miles of open country
We have stories still untold

[Chorus]
Carry the light, carry it home
Across the night, we are not alone
Follow the river, follow the sun
The journey has only begun

[Verse 2]
Through the rain we kept on walking
Through the wind we found a song
Every voice became a lantern
Every step has brought us on

[Chorus]
Carry the light, carry it home
Across the night, we are not alone
Follow the river, follow the sun
The journey has only begun

[Bridge]
When the road bends out of sight
We will find another light

[Final Chorus]
Carry the light, carry it home
Across the night, we are not alone
Follow the river, follow the sun
The journey has only begun

[Outro]
Carry it home
We are not alone
"""
song = SongConfig(
    style=STYLE, lyrics=LYRICS, max_duration=MAX_DURATION_SECONDS,
    mode=PLANNING_MODE, abc=CUSTOM_ABC, plan_only=PLAN_ONLY,
    max_abc_tokens=MAX_ABC_TOKENS, steps=ACOUSTIC_STEPS,
    semantic_cfg=SEMANTIC_CFG, abc_seed=ABC_SEED,
    semantic_seed=SEMANTIC_SEED, acoustic_seed=ACOUSTIC_SEED,
)
print(f"Maximum: {song.max_duration / 60:.2f} minutes; planning: {song.mode}")

**Generate.** This uses a private local API, so there is no tunnel to lose.
Keep this cell running. Stage status is printed as it changes; detailed
model progress goes to `server.log`. A run can take substantially longer
than the audio on T4; no speed claim has been established.

To stop: interrupt this cell. It sends an interrupt to the managed server.
Cancellation takes effect at native processing boundaries; wait for the
server to become idle before another request. `app.stop()` terminates only
this notebook's process if a GPU operation is stuck. Partial artifacts remain.

In [ ]:
app.start()
run_dir = app.run(song, timeout=2 * 60 * 60)

**Inspect the result.** `summary.json` distinguishes natural endings from
cutoffs and reports actual audio length. Natural ending means the model
emitted its end token; it does not prove lyrical or musical completeness.
Listen to the outro and every listed acoustic boundary. Seven/eight-minute
generations may have multiple independently synthesized acoustic chunks.
The native ComfyUI decoder also applies its standard loudness adjustment;
any additional PCM export gain is recorded explicitly.

In [ ]:
from IPython.display import Audio, display
print((run_dir / "summary.json").read_text())
if (run_dir / "audio.wav").exists():
    if json.loads((run_dir / "summary.json").read_text())["audio"]["duration_seconds"] <= 180:
        display(Audio(filename=str(run_dir / "audio.wav")))
    else:
        print("Long audio: download audio.flac/audio.wav from Kaggle's Output files panel.")
elif (run_dir / "score.abc").exists():
    print((run_dir / "score.abc").read_text())

**Recovery (optional).** A failed VAE decode can reuse `latents.safetensors`
without rerunning the score, semantic generation or acoustic sampler.
For a sampler failure, `RECOVERY_MODE = "tokens"` reuses completed token
stages and recomputes acoustic synthesis. Recovery checks inputs, runtime
identities and artifact checksums; it fails on a mismatch instead of silently
reusing the wrong song. Each attempt gets a new run directory.

For a previous-session ZIP, extract it under a new directory in
`/kaggle/working`, or attach its extracted files as a Kaggle input. Point
`RECOVER_RUN` at the folder containing `request.json`.

In [ ]:
RECOVER_RUN = None  # Example: "/kaggle/working/yue2/runs/20261023T120000Z-0123456789"
RECOVERY_MODE = "decode"  # "decode" or "tokens"
if RECOVER_RUN:
    app.start()
    if RECOVERY_MODE == "decode":
        run_dir = app.decode_saved(RECOVER_RUN, tile_size=256)
    elif RECOVERY_MODE == "tokens":
        previous = json.loads((Path(RECOVER_RUN) / "request.json").read_text())
        run_dir = app.run(SongConfig(**previous["song"]), resume_from=RECOVER_RUN)
    else:
        raise ValueError("Choose decode or tokens")

**Preserve the run and release GPU memory.** Set `MAKE_ZIP = True` to package
the selected run. Download it through Kaggle's Output files panel or save
a notebook version with outputs. Do not assume live working files persist
when a session ends. Setup caches and dependencies remain on disk; the
managed process is stopped below. To generate again, rerun the settings
and generation cells; the latter restarts the managed process.

In [ ]:
MAKE_ZIP = False
if MAKE_ZIP and "run_dir" in globals():
    print("Download this archive from Kaggle Output:", app.archive(run_dir))
app.stop()
print("Managed server stopped; generated files remain under", app.runs)